# Qwen2.5-3B — LoRA + SFT on Dolly 15k

Runs the pipeline from the repo on a Colab GPU.

**Before you start:** Runtime → Change runtime type → **T4 GPU** (or better).

Checkpoints go to Google Drive, so a disconnected session resumes
instead of starting over. Re-run this notebook top to bottom after a
drop and training continues from the last checkpoint.


## 1. Check the GPU


In [ ]:
!nvidia-smi

import torch
print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('bf16 supported:', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)


`bf16 supported: False` means a T4, which is fine — the code falls back to fp16 automatically.


## 2. Mount Drive

`/content` is wiped when the session ends. Checkpoints must live on Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/qwen2.5-3b-dolly-lora'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.environ['OUTPUT_DIR'] = OUTPUT_DIR
print('Checkpoints ->', OUTPUT_DIR)


## 3. Clone the repo


In [ ]:
BRANCH = 'Qwen-2.5-3B_finetuned_with-Databricks-Dolly-instruct-dataset'
REPO = 'https://github.com/MuzzamilRauf/LLM_Finetuning.git'

import os
if not os.path.exists('/content/LLM_Finetuning'):
    !git clone --branch $BRANCH $REPO /content/LLM_Finetuning
else:
    !cd /content/LLM_Finetuning && git pull

%cd /content/LLM_Finetuning


## 4. Install dependencies

torch is deliberately excluded — Colab's build is matched to its CUDA driver.


In [ ]:
!pip install -q -r requirements.txt

import transformers, trl, peft, datasets, torch
for m in (torch, transformers, datasets, trl, peft):
    print(f'{m.__name__:14} {m.__version__}')


Restart the runtime if pip asks, then re-run from step 2.


## 5. Build the dataset

Regenerated here rather than committed — deterministic on seed 42,
so the splits match the local ones exactly. Skips if already built.


In [ ]:
import os
if os.path.exists('data/processed/dolly-prepared'):
    print('Dataset already built, skipping.')
else:
    !python src/clean_data.py
    !python src/prepare_data.py


## 6. Smoke test

`MAX_TRAIN_SAMPLES = 100` in `src/config.py`. Confirm the CUDA path
works before committing to a long run.


In [ ]:
!python src/train.py


## 7. Full run

Only after step 6 passes. Edit `src/config.py`:

```python
MAX_TRAIN_SAMPLES = None   # all 13,496 examples
MAX_EVAL_SAMPLES = 200
EVAL_STEPS = 100
SAVE_STEPS = 100           # ~17 checkpoints, not 337
```

Raising `PER_DEVICE_TRAIN_BATCH_SIZE` to 4 (and dropping
`GRADIENT_ACCUMULATION_STEPS` to 2) uses the GPU far better than
the batch size of 1 the 16 GB Mac needed.


In [ ]:
!python src/train.py


## 8. Resuming after a disconnect

Re-run steps 2 → 4 → 7. `find_checkpoint()` picks up the newest
checkpoint on Drive on its own; no flags to set.


In [ ]:
!ls -la $OUTPUT_DIR


## 9. Try the adapter


In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE = 'Qwen/Qwen2.5-3B'
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float16, device_map='auto')
model = PeftModel.from_pretrained(model, os.environ['OUTPUT_DIR'])
model.eval()

messages = [{'role': 'user', 'content': 'Explain photosynthesis in two sentences.'}]
inputs = tok.apply_chat_template(messages, add_generation_prompt=True,
                                 return_tensors='pt').to(model.device)
out = model.generate(inputs, max_new_tokens=200, do_sample=False)
print(tok.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))
